# Day 4, Part 3: From Similarity to Topics

**MIDAS Summer Academy**

In Part 1 you used pre-trained models off the shelf. In Part 2 you used a large language model to extract structured data from text. This notebook works with the representation underneath both: the **embedding**.

In this notebook you will:

1. Compute embeddings for individual sentences and measure the similarity between them with a **dot product**.
2. Embed a corpus of ~2,000 research abstracts.
3. Use **BERTopic** to discover topics in the corpus without labels or supervision.

---
## Setup

Run this once. The install takes a minute or two.

In [ ]:
!pip install -q bertopic sentence-transformers

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=3, suppress=True)  # readable arrays
print("Ready.")

---
## 1. An embedding is an array of numbers

An **embedding model** maps a piece of text to a fixed-length vector. Embedding models are trained so that texts with similar meaning map to vectors that point in similar directions.

We will use `all-MiniLM-L6-v2`, a small sentence embedding model. It maps any sentence to a **384-dimensional** vector.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # ~80 MB download

sentences = [
    "The cat sat on the warm windowsill.",
    "A kitten napped in the sunny window.",
    "The stock market fell sharply on Tuesday.",
    "Investors sold shares as prices dropped.",
    "I added fresh basil to the tomato sauce.",
]

embeddings = model.encode(sentences)
print("Shape:", embeddings.shape, " (5 sentences x 384 numbers each)")

The next cell prints part of one embedding. It is a vector of 384 floating-point numbers.

In [ ]:
print("Embedding for:", sentences[0])
print("First 12 of its 384 numbers:")
print(embeddings[0][:12])

# These vectors are already normalized to length 1 (unit vectors).
print("\nLength (norm) of this vector:", np.linalg.norm(embeddings[0]).round(3))

### Measuring similarity with a dot product

For two vectors, the **dot product** is

$$a \cdot b = \sum_i a_i b_i$$

**Cosine similarity** is the dot product divided by the product of the two vector magnitudes. The embeddings from this model are normalized to length 1 (**unit vectors**), so the dot product and the cosine similarity are equal. The value ranges from -1 to 1; higher values indicate greater similarity.

The next cell computes the dot product for every pair of sentences.

In [ ]:
# Compute the dot product for every pair of sentences
n = len(sentences)
similarity = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        similarity[i, j] = np.dot(embeddings[i], embeddings[j])

labels = [f"S{i}" for i in range(n)]
print(pd.DataFrame(similarity, index=labels, columns=labels).round(2))

print()
for i, s in enumerate(sentences):
    print(f"S{i}: {s}")

### Find the closest pair

The next cell finds the two *different* sentences with the highest dot product (the largest off-diagonal entry in the matrix).

In [ ]:
# Ignore the diagonal (every sentence is identical to itself)
sim = similarity.copy()
np.fill_diagonal(sim, -np.inf)

i, j = np.unravel_index(np.argmax(sim), sim.shape)
print(f"Closest pair (score {sim[i, j]:.2f}):")
print(f"  {sentences[i]}")
print(f"  {sentences[j]}")

Compare the result against the full similarity matrix. Which pairs of sentences score high with each other, and which score low? Does the pattern match the meanings of the sentences? Note that the model receives only the text; it has no other information about which sentences are related.

### Exercise 1

Add a sixth sentence to the `sentences` list: a *paraphrase* of an existing sentence (different words, same meaning). Re-embed and rebuild the similarity matrix. Does the new sentence score highest with the sentence it paraphrases?

In [ ]:
# YOUR CODE HERE
# Copy the sentences list from above, append one new sentence, then re-embed
# and rebuild the similarity matrix.


---
## 2. Scaling up: 2,000 abstracts

The next dataset contains **~2,000 recent arXiv abstracts** drawn from six fields (astrophysics, machine learning, neuroscience, materials science, econometrics, atmospheric science). A corpus this size is too large to read directly. Instead, we embed every abstract and analyze the geometry of the resulting vectors.

In [ ]:
import os

LOCAL = "arxiv_abstracts/abstracts.csv"
URL = "https://gist.githubusercontent.com/elleobrien/16b86d0447fbe49a5fce5172e670150c/raw/abstracts.csv"

df = pd.read_csv(LOCAL if os.path.exists(LOCAL) else URL)
print(f"Loaded {len(df)} abstracts.")
df["field"].value_counts()

The `field` column is the ground truth: the arXiv category each abstract came from. BERTopic does not use this column. We keep it to check, at the end, whether the discovered topics align with the actual fields.

---
## 3. Discover topics with BERTopic

**BERTopic** is a topic modeling library that chains four steps:

1. **Embed** every abstract, as in Section 1.
2. **Reduce** the 384 dimensions to a smaller number, preserving local structure (an algorithm called UMAP).
3. **Cluster** the reduced vectors; each cluster becomes a topic (an algorithm called HDBSCAN).
4. **Label** each cluster with the words most distinctive to it.

BERTopic runs steps 2-4 internally with reasonable defaults. We supply only the embedding model and a few settings. The [algorithm section of the BERTopic documentation](https://maartengr.github.io/BERTopic/algorithm/algorithm.html) has a visual walkthrough of each step.

Embed the abstracts once and reuse the result; embedding is the slowest step.

In [ ]:
abstracts = df["abstract"].tolist()

# ~30-60s on Colab CPU, a few seconds on a GPU runtime.
abstract_embeddings = model.encode(abstracts, show_progress_bar=True)
print("Embedded:", abstract_embeddings.shape)

Now build the model. Two arguments matter here:

- `min_topic_size` is the minimum number of documents a cluster needs to count as a topic. Smaller values produce more, finer topics; larger values produce fewer, broader topics.
- `vectorizer_model` removes common English words (stopwords) before choosing topic keywords, so the labels are built from meaningful terms.

The dimensionality reduction step is stochastic, so the exact topics and their numbering can vary slightly between runs.

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

MIN_TOPIC_SIZE = 10

topic_model = BERTopic(
    embedding_model=model,
    min_topic_size=MIN_TOPIC_SIZE,
    vectorizer_model=CountVectorizer(stop_words="english"),
)

# Pass the precomputed embeddings so BERTopic does not recompute them.
topics, _ = topic_model.fit_transform(abstracts, embeddings=abstract_embeddings)
print("Found", len(set(topics)) - (1 if -1 in topics else 0), "topics.")

### Inspect the topics

`get_topic_info()` lists every topic, its size, and its top keywords.

In [ ]:
topic_model.get_topic_info()[["Topic", "Count", "Name"]]

**About Topic -1.** BERTopic assigns documents that do not fit confidently into any cluster to topic `-1`. This is the outlier group, not a real topic. HDBSCAN does not force every document into a cluster, so some outliers are expected.

The next cell prints the defining words for a single topic:

In [ ]:
# Pick the first real topic (skip -1)
first_topic = topic_model.get_topic_info().query("Topic != -1")["Topic"].iloc[0]
print("Top words for topic", first_topic, ":")
for word, score in topic_model.get_topic(first_topic):
    print(f"  {word:20s} {score:.3f}")

### Did the topics recover the fields?

Cross-tabulate the discovered topics against the ground-truth `field` column. Each row is a topic; each column is a field. Examine the table: which fields map cleanly onto one topic, and which are split across several? What about those fields might explain the difference?

In [ ]:
df["topic"] = topics
# Rows are discovered topics, columns are true fields.
crosstab = pd.crosstab(df["topic"], df["field"])
crosstab

---
## 4. Explore the topic space

BERTopic includes interactive visualizations. The **intertopic distance map** below projects the topics into two dimensions: each circle is a topic, circle size is the number of abstracts in it, and nearby circles contain semantically related abstracts.

In [ ]:
topic_model.visualize_topics()

### Search by meaning

`find_topics` embeds a query string and returns the topics whose contents are closest to it in embedding space. This is the same similarity measure from Section 1, applied as a search over topics.

In [ ]:
query = "exploding stars and supernovae"
found, scores = topic_model.find_topics(query, top_n=3)

print(f'Topics most similar to: "{query}"\n')
for t, s in zip(found, scores):
    words = ", ".join(w for w, _ in topic_model.get_topic(t)[:6])
    print(f"  Topic {t} (score {s:.2f}): {words}")

### Exercise 2

1. Call `find_topics` with a phrase from your own research area. Do the returned topics make sense?
2. Change `MIN_TOPIC_SIZE` (try `5`, then `25`) and re-run Section 3. How does the number of topics change? There is no single correct value; the choice depends on the granularity your analysis needs.

In [ ]:
# YOUR CODE HERE
# query = "..."
# found, scores = topic_model.find_topics(query, top_n=3)


---
## Bonus: a science-specific embedding model

`all-MiniLM-L6-v2` was trained on general text from the web. **SPECTER** is an embedding model built for scientific papers. It starts from **SciBERT**, a BERT variant pre-trained on scientific text, and is fine-tuned on citation data: papers that cite each other are trained to have similar embeddings. It maps each document to a **768-dimensional** vector.

Re-embed the abstracts with SPECTER and rerun BERTopic with the same settings. The model is larger than MiniLM (~440 MB download), and embedding takes several minutes on a CPU runtime.

In [ ]:
specter = SentenceTransformer("sentence-transformers/allenai-specter")   # ~440 MB download

# Several minutes on a CPU runtime; much faster on a GPU runtime.
specter_embeddings = specter.encode(abstracts, show_progress_bar=True)
print("Embedded:", specter_embeddings.shape)

In [ ]:
specter_topic_model = BERTopic(
    embedding_model=specter,
    min_topic_size=MIN_TOPIC_SIZE,
    vectorizer_model=CountVectorizer(stop_words="english"),
)

specter_topics, _ = specter_topic_model.fit_transform(abstracts, embeddings=specter_embeddings)
print("Found", len(set(specter_topics)) - (1 if -1 in specter_topics else 0), "topics.")
specter_topic_model.get_topic_info()[["Topic", "Count", "Name"]]

In [ ]:
# Rows are discovered topics, columns are true fields.
pd.crosstab(pd.Series(specter_topics, name="topic"), df["field"])

Compare this cross-tabulation with the one from Section 3. Does SPECTER separate machine learning from neuroscience differently than MiniLM? Compare the topic keywords as well: do the two models organize the corpus around different distinctions? Domain-specific pre-training changes which similarities the model emphasizes, so there is no guaranteed winner.

---
## Wrap-up

| Step in this notebook | Concept |
|---|---|
| Embedded sentences into 384-dimensional vectors | Text represented as vectors in a high-dimensional space |
| Dot product of unit vectors | Cosine similarity |
| UMAP | Dimensionality reduction that preserves local structure |
| HDBSCAN | Clustering nearby vectors into groups |
| `find_topics` | Similarity search in embedding space |

Once data is embedded, the same toolkit (similarity, clustering, visualization, search) applies regardless of the original data type: text, images, sequences, or signals.

> **Discussion:** What corpus from your own work could you embed and cluster this way? What structure would you look for that is not visible by reading?